In [150]:
# -*- coding: utf-8 -*-
"""
Filtrage thématique du catalogue data.gouv.fr.

- Lit `type-donnees-mobi-mc.csv` : dictionnaire {sous-thème: termes descriptifs}
  pour un thème donné, et liste des mots-clés du thème (colonne "Thème").
- Filtre `datasets_mobi.csv` : garde les lignes dont "title", "tags" ou
  "description" contient au moins un des mots-clés du thème.
"""

# code généré par Mistral produire une fonction python qui prend en entrée le nom du thème, et qui
# 1) lit le tableau des descriptions des sous-thèmes type-donnees-mobi-mc.csv, en extrait un dict des termes descriptifs (colonne "termes") associé à chaque sous-thème du thème donné en entrée
# 2) extrait du tableau extrait du catalogue de data.gouv.fr, datasets_mobi.csv, un dataframe gardant uniquement les lignes dont la colonne "title", "tags", ou "description" contient l'un des mots-clés de la colonne "mots-clés-thème"

import re
import unicodedata
from pathlib import Path

import pandas as pd

# --------------------------------------------------------------------------- #
# Paramètres
# --------------------------------------------------------------------------- #

CHEMIN_SOUS_THEMES = "../data/type-donnees-mobi-mc.csv"   # Catégorie;Thème;Sous-thème;Mots-clés;Termes
CHEMIN_DATASETS = "../data/datasets_mobi.csv"             # export catalogue data.gouv.fr (séparateur ";")
COLONNES_TEXTE = ["title", "tags", "description"]

def _normaliser(texte: str) -> str:
    """Minuscules + suppression des accents (les CSV mélangent avec/sans accents)."""
    if texte is None or (isinstance(texte, float) and pd.isna(texte)):
        return ""
    texte = str(texte).lower()
    texte = unicodedata.normalize("NFKD", texte)
    texte = "".join(c for c in texte if not unicodedata.combining(c))
    return texte

def _mot_de_passe(texte_normalise: str, mot_cle_normalise: str) -> bool:
    """
    Vrai si le mot-clé apparaît dans le texte.
    - Insensible à la casse et aux accents (les textes sont déjà normalisés).
    - Le mot-clé "comptage" doit matcher "comptages" (pluriel) : on accepte
      une terminaison libre de 0 à 2 caractères alphabétiques.
    - On évite les faux positifs de sous-chaîne (ex. "bus" dans "bush")
      en exigeant une frontière de mot au début du mot-clé.
    """
    motif = r"\b" + re.escape(mot_cle_normalise) + r"[a-z]{0,2}\b"
    return re.search(motif, texte_normalise) is not None



In [167]:
# --------------------------------------------------------------------------- #
# Étape 1 : lecture de la taxonomie
# --------------------------------------------------------------------------- #

COL_MOTS_CLES_THEME = "Mots-clés thème"

def charger_taxonomie(chemin: str | Path = CHEMIN_SOUS_THEMES) -> pd.DataFrame:
    """
    Charge le tableau des sous-thèmes (séparateur ',', colonnes en str).

    Accepte les deux formats :
    - nouveau : Catégorie;Thème;Mots-clés thème;Sous-thème;Mots-clés;Termes
      (la colonne "Thème" contient le nom du thème, la colonne
      "Mots-clés thème" ses termes de recherche)
    - ancien   : la colonne "Thème" contenait à la fois le nom et les termes,
      séparés par des virgules (ex. "Transport collectif, bus, train, tram").
    """
    taxo = pd.read_csv(chemin, sep=",", dtype=str)
    taxo.columns = [c.strip().strip("\ufeff") for c in taxo.columns]
    # Normalise les variantes de nommage de la colonne mots-clés du thème
    renommages = {c: COL_MOTS_CLES_THEME for c in taxo.columns
                  if _normaliser(c) in ("mots-cles theme", "mots-cles-theme",
                                        "mot-cles theme")}
    taxo = taxo.rename(columns=renommages)
    return taxo.fillna("")


def _lignes_du_theme(taxo: pd.DataFrame, theme: str) -> pd.DataFrame:
    """Lignes de la taxonomie dont le thème (nouveau ou ancien format) correspond."""
    theme_norm = _normaliser(theme)
    masque = taxo["Thème"].apply(
        lambda cell: theme_norm in [_normaliser(t) for t in cell.split(",")]
    )
    return taxo[masque]


def extraire_theme(
    taxo: pd.DataFrame,
    theme: str,
) -> tuple[dict[str, str], list[str]]:
    """
    Pour un thème donné (ex. "Stationnement", "Vélo", "Trafic routier") :

    1. Sélectionne les lignes du tableau dont la colonne "Thème" correspond
       (au nouveau format, la cellule contient le nom du thème seul ; au
       format ancien, elle peut lister plusieurs variantes séparées par
       des virgules : "Voirie, Route" — les deux sont gérés).
    2. Renvoie :
       - dictionnaire {sous-thème: termes descriptifs} (colonne "Termes")
       - liste des mots-clés du thème :
         * nouveau format : contenu de la colonne "Mots-clés thème"
           (ex. "Transport collectif, transport public, bus, train, tram")
         * ancien format (colonne absente) : termes extraits de la cellule
           "Thème" elle-même.

    L'association au thème se fait sans accents ni casse.
    """

    ## ATTENTION dans le code actuel, la recherche du thème est sensible à la casse
    
    lignes = _lignes_du_theme(taxo, theme)
    if lignes.empty:
        raise ValueError(
            f"Thème '{theme}' introuvable dans la colonne 'Thème'. "
            f"Thèmes disponibles : {sorted(taxo['Thème'].unique())}"
        )

    # Dict des sous-thèmes du thème -> termes descriptifs
    sous_themes = {
        row["Sous-thème"].strip(): row["Termes"].strip()
        for _, row in lignes.iterrows()
    }
    
    # Mots-clés du thème
    x = taxo[taxo['Thème'] == theme]['Mots-clés thème'].reset_index(drop=True)
    print(x)
    print(x[0])
    mc = [item.strip() for item in x[0].split(',')]
    mctheme = dict()
    mctheme[theme] = mc
    return sous_themes, mctheme


In [146]:
# --------------------------------------------------------------------------- #
# Étape 2 : filtrage du catalogue
# --------------------------------------------------------------------------- #

def charger_catalogue(chemin: str | Path = CHEMIN_DATASETS) -> pd.DataFrame:
    """Charge l'extrait du catalogue (séparateur ';', 1re colonne = index)."""
    cat = pd.read_csv(chemin, sep=";", index_col=0)
    return cat


def filtrer_datasets_par_theme(
    theme: str,
    chemin_sous_themes: str | Path = CHEMIN_SOUS_THEMES,
    chemin_catalogue: str | Path = CHEMIN_DATASETS,
    inclure_mots_cles_sous_themes: bool = True,
    verbose: bool = True,
) -> tuple[dict[str, str], pd.DataFrame]:
    """
    Pipeline complet : renvoie
      (dict {sous-thème: termes descriptifs}, DataFrame des datasets filtrés)

    Paramètres
    ----------
    theme : str
        Nom du thème, ex. "Stationnement", "Vélo", "Trafic routier",
        "Transport collectif"... (matching sans accents ni casse).
    inclure_mots_cles_sous_themes : bool
        Si True, les mots-clés de chaque sous-thème (colonne "Mots-clés")
        sont ajoutés aux mots-clés de filtrage, en plus des termes du thème.
        Le filtrage est alors plus large mais moins discriminant : les
        sous-thèmes sont justement censés être départagés plus finement
        par l'étape embeddings.
    """
    taxo = charger_taxonomie(chemin_sous_themes)
    sous_themes, themes = extraire_theme(taxo, theme)

    # Déduplication, normalisation des mots-clés
    ### à revoir !!
    mots_cles_norm = sorted({_normaliser(m) for m in themes[theme] if m})
    print(mots_cles_norm)

    catalogue = charger_catalogue(chemin_catalogue)

    # Texte de recherche : concaténation des colonnes textuelles, normalisée
    catalogue[COLONNES_TEXTE] = catalogue[COLONNES_TEXTE].fillna('')
    texte_global = catalogue[COLONNES_TEXTE].agg(" ".join, axis=1).apply(_normaliser)

    # Un dataset est retenu si au moins un mot-clé du thème apparaît
    motifs = [r"\b" + re.escape(m) + r"[a-z]{0,2}\b" for m in mots_cles_norm]
    regex_global = re.compile("|".join(motifs))
    masque = texte_global.apply(lambda t: bool(regex_global.search(t)))

    resultat = catalogue[masque].copy()

    if verbose:
        print(f"Thème                 : {theme}")
        print(f"Sous-thèmes trouvés   : {len(sous_themes)}")
        for st, termes in sous_themes.items():
            print(f"  - {st} : {termes[:70]}...")
        print(f"Mots-clés de filtrage : {mots_cles_norm}")
        print(f"Datasets              : {len(resultat)} / {len(catalogue)}")

    return sous_themes, resultat


In [151]:
catalogue = charger_catalogue()

In [152]:
taxo=charger_taxonomie()

In [169]:
sous_themes, mctheme = extraire_theme(taxo, 'Voirie')

0    Voirie, Route
1    Voirie, Route
2    Voirie, Route
Name: Mots-clés thème, dtype: str
Voirie, Route


In [148]:
taxo = charger_taxonomie()